## Contents
1. Setup & Imports  
2. Data Generation  
3. Graph Construction  
4. Model Definitions  
5. Training  
6. Evaluation & Shock Test  
7. λ Sensitivity Sweep  
8. Plots (1–5)  
9. Final Summary  

In [ ]:
# Cell 1 — Setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

try:
    from torch_geometric.nn import GATConv
    HAS_PYG = True
except ImportError:
    HAS_PYG = False

print(f'PyTorch : {torch.__version__}')
print(f'PyG     : {"available" if HAS_PYG else "NOT found — using fallback GAT"}')

In [ ]:
# Cell 2 — Hyperparameters
T_STEPS    = 300
N_LEVELS   = 10
HP         = dict(in_features=2, hidden_dim=32, out_features=2, heads=4, dropout=0.1)
EPOCHS     = 100
LR         = 1e-3
LAMBDA     = 0.1
SEED       = 42
COLORS     = {'baseline': '#E05C5C', 'physics': '#4A90D9'}

torch.manual_seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({
    'font.family':      'monospace',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.25,
    'figure.dpi':       110,
})

In [ ]:
# Cell 3 — Data
from liqflow.data import build_lob_tensor, train_test_split

lob, meta = build_lob_tensor(T=T_STEPS, P=N_LEVELS, seed=SEED)
train_t, test_t = train_test_split(lob)

print(f'Tensor shape : {tuple(lob.shape)}')
print(f'Train        : {tuple(train_t.shape)}')
print(f'Test         : {tuple(test_t.shape)}')
print(f'Price grid   : {np.round(meta["prices"], 2)}')

In [ ]:
# Cell 4 — Graph construction
from liqflow.graph import build_spatial_graph, build_spatiotemporal_graph

snap_st = build_spatiotemporal_graph(lob, t=5, W=5, prices=meta['prices'])
snap_sp = build_spatial_graph(lob, t=0, prices=meta['prices'])

print(f'Spatio-temporal graph  nodes={snap_st.x.shape[0]}  edges={snap_st.edge_index.shape[1]}')
print(f'  spatial={snap_st.num_spatial}  temporal={snap_st.num_temporal}')
print(f'Spatial-only graph     nodes={snap_sp.x.shape[0]}  edges={snap_sp.edge_index.shape[1]}')

In [ ]:
# Cell 5 — Models
from liqflow.models import BaselineGAT, PhysicsGAT

baseline_model = BaselineGAT(**HP)
physics_model  = PhysicsGAT(**HP, lam=LAMBDA, D_init=0.1, learnable_D=True)

print(f'BaselineGAT params : {sum(p.numel() for p in baseline_model.parameters()):,}')
print(f'PhysicsGAT  params : {sum(p.numel() for p in physics_model.parameters()):,}')

In [ ]:
# Cell 6 — Training
from liqflow.train import train_baseline, train_physics

print('Training Baseline GAT …')
base_hist = train_baseline(baseline_model, train_t, test_t, epochs=EPOCHS, verbose=True)

print(f'\nTraining Physics GAT (λ={LAMBDA}) …')
phys_hist = train_physics(physics_model, train_t, test_t, epochs=EPOCHS, verbose=True)

In [ ]:
# Cell 7 — Evaluation
from liqflow.evaluate import compute_metrics, run_shock_test, lambda_sensitivity

eval_base = compute_metrics(baseline_model, test_t)
eval_phys = compute_metrics(physics_model,  test_t)
eval_out  = {'baseline': eval_base, 'physics': eval_phys}

shock_out = run_shock_test(baseline_model, physics_model, lob,
                           t0=100, shock_level=4, magnitude=5.0, steps=5)
print('Evaluation complete.')

In [ ]:
# Cell 8 — Plot 1: Loss curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Plot 1 — Training Loss Curves', fontweight='bold')
for ax, split in zip(axes, ['train', 'val']):
    ax.plot(base_hist[split], color=COLORS['baseline'], lw=1.8, label='Baseline GAT')
    ax.plot(phys_hist[split], color=COLORS['physics'],  lw=1.8, label='Physics GAT')
    ax.set_title(f'{split.capitalize()} Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 9 — Plot 2: MSE comparison
keys   = ['mse_total', 'mse_volume', 'mse_imbalance']
labels = ['Total MSE', 'Volume MSE', 'Imbalance MSE']
x = np.arange(len(keys)); w = 0.32

fig, ax = plt.subplots(figsize=(8, 4.5))
fig.suptitle('Plot 2 — Test MSE Comparison', fontweight='bold')
for i, (name, color) in enumerate([('baseline', COLORS['baseline']),
                                    ('physics',  COLORS['physics'])]):
    vals = [eval_out[name][k] for k in keys]
    bars = ax.bar(x + i*w, vals, w, color=color, label=name.capitalize(), alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x + w/2); ax.set_xticklabels(labels)
ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 10 — Plot 3: Shock heatmaps
prices = meta['prices']; P = lob.shape[1]
fig = plt.figure(figsize=(15, 5))
fig.suptitle('Plot 3 — Shock Propagation Heatmaps (Volume)', fontweight='bold')
gs = gridspec.GridSpec(1, 3, wspace=0.35)
panels = {
    'True (shocked)': shock_out['true'][..., 0].numpy(),
    'Baseline GAT':   shock_out['baseline'][..., 0].numpy(),
    'Physics GAT':    shock_out['physics'][..., 0].numpy(),
}
all_vals = np.concatenate([v.ravel() for v in panels.values()])
vmin, vmax = all_vals.min(), all_vals.max()
norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax) if vmin < 0 < vmax else None
for idx, (title, data) in enumerate(panels.items()):
    ax = fig.add_subplot(gs[idx])
    im = ax.imshow(data.T, aspect='auto', origin='lower',
                   norm=norm, cmap='RdYlBu_r', interpolation='nearest')
    ax.set_title(title, fontsize=10); ax.set_xlabel('Step after shock')
    ax.set_ylabel('Price level'); ax.set_yticks(range(P))
    ax.set_yticklabels([f'{p:.1f}' for p in prices], fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

In [ ]:
# Cell 11 — Plot 4: Residuals over time
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
fig.suptitle('Plot 4 — Prediction Residual Magnitude Over Time', fontweight='bold')
for ax, fi, fname in zip(axes, [0, 1], ['Volume', 'Bid-Ask Imbalance']):
    for name, color in COLORS.items():
        preds   = eval_out[name]['preds']
        targets = eval_out[name]['targets']
        res  = (preds[..., fi] - targets[..., fi]).abs().mean(dim=1).numpy()
        ax.plot(res, color=color, lw=1.3, alpha=0.6, label=name.capitalize())
        k = 10; roll = np.convolve(res, np.ones(k)/k, mode='valid')
        ax.plot(range(k-1, len(res)), roll, color=color, lw=2.2, linestyle='--')
    ax.set_ylabel(f'|Residual| — {fname}'); ax.legend(fontsize=8)
axes[-1].set_xlabel('Time step')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 12 — λ sweep (optional — takes ~5 min)
RUN_SWEEP = False   # set True to run

if RUN_SWEEP:
    LAMBDAS     = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
    lam_results = lambda_sensitivity(train_t, test_t, LAMBDAS, epochs=50)
    lams = sorted(lam_results.keys()); mses = [lam_results[l] for l in lams]
    base_mse = eval_out['baseline']['mse_total']
    fig, ax = plt.subplots(figsize=(8, 4.5))
    fig.suptitle('Plot 5 — λ Sensitivity', fontweight='bold')
    ax.semilogx(lams, mses, 'o-', color=COLORS['physics'], lw=2, markersize=8)
    ax.axhline(base_mse, color=COLORS['baseline'], lw=1.8, linestyle='--',
               label=f'Baseline MSE={base_mse:.4f}')
    ax.set_xlabel('λ'); ax.set_ylabel('Test MSE'); ax.legend()
    plt.tight_layout(); plt.show()
else:
    print('Set RUN_SWEEP=True to execute the λ sweep.')

In [ ]:
# Cell 13 — Final summary
for name in ['baseline', 'physics']:
    m = eval_out[name]; s = shock_out['metrics'][name]
    print(f'\n{name.upper()}')
    print(f' Test MSE (total)     : {m["mse_total"]:.5f}')
    print(f' Test MSE (volume)    : {m["mse_volume"]:.5f}')
    print(f' Test MSE (imbalance) : {m["mse_imbalance"]:.5f}')
    print(f' Shock recovery err   : {s["shock_recovery_error"]:.5f}')
    print(f' Stability std        : {s["stability_std"]:.5f}')

bm = eval_out['baseline']; pm = eval_out['physics']
print(f'\n DELTA (physics − baseline)')
print(f'  ΔMSE           : {pm["mse_total"]-bm["mse_total"]:+.5f}')
print(f'  Δstability std : {shock_out["metrics"]["physics"]["stability_std"]-shock_out["metrics"]["baseline"]["stability_std"]:+.5f}')
print(f'  Learned D      : {physics_model.physics.D.item():.4f}')